# Model Building Code

This notebook trains the five candidate architectures and writes the checkpoint files used by `Project_Report.ipynb`. It is included for reproducibility, but the report notebook loads the saved checkpoints directly so it can run much faster.


In [ ]:
import random
from collections import Counter
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image as PILImage
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler, random_split
from torchvision import models, transforms
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT
DATA_DIR = DATA_ROOT / "Data"
CHECKPOINT_DIR = DATA_ROOT / "model_checkpoints"
CHECKPOINT_DIR.mkdir(exist_ok=True)

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}
CLASS_NAMES = ["B_Cells", "Macrophages", "T_Cells"]
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
SEED = 3888

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Device: {device}")
print(f"Data directory: {DATA_DIR}")
print(f"Checkpoint directory: {CHECKPOINT_DIR}")


In [ ]:
CLASS_PATHS = {
    "B_Cells": [DATA_DIR / "B_Cells"],
    "Macrophages": [DATA_DIR / "Macrophages"],
    "T_Cells": [
        DATA_DIR / "CD4+_T_Cells",
        DATA_DIR / "CD8+_T_Cells",
    ],
}


def build_sample_records():
    records = []
    for class_name in CLASS_NAMES:
        for folder in CLASS_PATHS[class_name]:
            if not folder.exists():
                print(f"Warning: missing folder skipped: {folder}")
                continue

            for file in sorted(folder.iterdir()):
                if file.is_file() and file.suffix.lower() in IMAGE_EXTS:
                    records.append((file, CLASS_TO_IDX[class_name]))
    return records


class ImmuneCellDataset(Dataset):
    def __init__(self, records, classes):
        self.records = list(records)
        self.classes = list(classes)
        self.targets = [label for _, label in self.records]

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        image_path, label = self.records[index]
        image = PILImage.open(image_path).convert("RGB")
        return image, label


class TransformSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
        self.classes = getattr(subset.dataset, "classes", CLASS_NAMES)

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, index):
        image, label = self.subset[index]
        if self.transform is not None:
            image = self.transform(image)
        return image, label


augmentation = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomResizedCrop(size=224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(45),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

base_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

sample_records = build_sample_records()
if len(sample_records) == 0:
    raise FileNotFoundError("No image files were found. Check that the Data folder is included beside this notebook.")

base_dataset = ImmuneCellDataset(sample_records, CLASS_NAMES)

train_size = int(len(base_dataset) * 0.60)
val_size = int(len(base_dataset) * 0.20)
test_size = len(base_dataset) - train_size - val_size

generator = torch.Generator().manual_seed(42)
train_subset, val_subset, test_subset = random_split(
    base_dataset,
    [train_size, val_size, test_size],
    generator=generator,
)

train_dataset = TransformSubset(train_subset, augmentation)
val_dataset = TransformSubset(val_subset, base_transform)
test_dataset = TransformSubset(test_subset, base_transform)

train_labels = [base_dataset.targets[idx] for idx in train_subset.indices]
class_counts_for_weights = Counter(train_labels)
weights_per_class = {
    class_idx: 1.0 / count
    for class_idx, count in class_counts_for_weights.items()
}
sample_weights = [weights_per_class[label] for label in train_labels]
weighted_sampler_train = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True,
)


def get_custom_dataloaders(target_batch_size):
    train_loader = DataLoader(
        train_dataset,
        batch_size=target_batch_size,
        sampler=weighted_sampler_train,
        num_workers=0,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=target_batch_size,
        shuffle=False,
        num_workers=0,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=target_batch_size,
        shuffle=False,
        num_workers=0,
    )
    return train_loader, val_loader, test_loader


pd.DataFrame(
    {
        "Split": ["Train", "Validation", "Test"],
        "Images": [train_size, val_size, test_size],
    }
)


In [ ]:
def amp_context():
    if device.type == "cuda":
        return torch.amp.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()


def make_grad_scaler():
    return torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))


def clone_state_dict(model):
    return {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}


def set_trainable(module, trainable):
    for parameter in module.parameters():
        parameter.requires_grad = trainable


def run_epoch(model, loader, criterion, optimizer=None, scaler=None, phase_label="Train"):
    is_training = optimizer is not None
    model.train(is_training)
    running_loss = 0.0
    correct = 0
    total = 0

    context = torch.enable_grad() if is_training else torch.no_grad()
    with context:
        for images, labels in tqdm(loader, desc=phase_label, leave=False):
            images = images.to(device)
            labels = labels.to(device)

            if is_training:
                optimizer.zero_grad()

            with amp_context():
                outputs = model(images)
                loss = criterion(outputs, labels)

            if is_training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

            running_loss += loss.item()
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)

    return {
        "loss": running_loss / max(len(loader), 1),
        "accuracy": 100 * correct / max(total, 1),
    }


def train_two_phase_model(
    model,
    head_modules,
    model_name,
    checkpoint_name,
    batch_size,
    phase1_lr,
    phase2_lr,
    phase1_epochs=3,
    phase2_epochs=5,
    weight_decay_phase1=1e-3,
    weight_decay_phase2=1e-4,
    patience=4,
):
    train_loader, val_loader, _ = get_custom_dataloaders(batch_size)
    model = model.to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler = make_grad_scaler()

    print(f"\n[{model_name}] Phase 1: classifier head only")
    set_trainable(model, False)
    for head_module in head_modules:
        set_trainable(head_module, True)

    optimizer = torch.optim.Adam(
        filter(lambda parameter: parameter.requires_grad, model.parameters()),
        lr=phase1_lr,
        weight_decay=weight_decay_phase1,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=phase1_epochs)

    best_val_acc = -np.inf
    best_state = clone_state_dict(model)

    for epoch in range(phase1_epochs):
        train_metrics = run_epoch(
            model,
            train_loader,
            criterion,
            optimizer=optimizer,
            scaler=scaler,
            phase_label=f"[{model_name} P1] Epoch {epoch + 1}/{phase1_epochs} Training",
        )
        val_metrics = run_epoch(
            model,
            val_loader,
            criterion,
            phase_label=f"[{model_name} P1] Epoch {epoch + 1}/{phase1_epochs} Validation",
        )
        scheduler.step()

        print(
            f"  Epoch {epoch + 1:02d}/{phase1_epochs} | "
            f"Train: {train_metrics['accuracy']:.1f}% | "
            f"Val: {val_metrics['accuracy']:.1f}%"
        )

        if val_metrics["accuracy"] > best_val_acc:
            best_val_acc = val_metrics["accuracy"]
            best_state = clone_state_dict(model)

    print(f"\n[{model_name}] Phase 2: full fine-tuning")
    set_trainable(model, True)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=phase2_lr,
        weight_decay=weight_decay_phase2,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=phase2_epochs)
    stale_epochs = 0

    for epoch in range(phase2_epochs):
        train_metrics = run_epoch(
            model,
            train_loader,
            criterion,
            optimizer=optimizer,
            scaler=scaler,
            phase_label=f"[{model_name} P2] Epoch {epoch + 1}/{phase2_epochs} Training",
        )
        val_metrics = run_epoch(
            model,
            val_loader,
            criterion,
            phase_label=f"[{model_name} P2] Epoch {epoch + 1}/{phase2_epochs} Validation",
        )
        scheduler.step()

        print(
            f"  Epoch {epoch + 1:02d}/{phase2_epochs} | "
            f"Train: {train_metrics['accuracy']:.1f}% | "
            f"Val: {val_metrics['accuracy']:.1f}%"
        )

        if val_metrics["accuracy"] > best_val_acc:
            best_val_acc = val_metrics["accuracy"]
            best_state = clone_state_dict(model)
            stale_epochs = 0
        else:
            stale_epochs += 1
            if stale_epochs >= patience:
                print(f"  Early stopping triggered at epoch {epoch + 1}")
                break

    model.load_state_dict(best_state)
    checkpoint_path = CHECKPOINT_DIR / checkpoint_name
    torch.save(best_state, checkpoint_path)
    model = model.to(device)
    model.eval()
    print(f"{model_name} best validation accuracy: {best_val_acc:.2f}%")
    print(f"Saved checkpoint: {checkpoint_path}")
    return model


In [ ]:
class CellResNet50(nn.Module):
    def __init__(self):
        super().__init__()
        base_model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.features = nn.Sequential(
            base_model.conv1,
            base_model.bn1,
            base_model.relu,
            base_model.maxpool,
            base_model.layer1,
            base_model.layer2,
            base_model.layer3,
            base_model.layer4,
        )
        self.avgpool = base_model.avgpool
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(2048, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.3),
            nn.Linear(256, len(CLASS_NAMES)),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


resnet = CellResNet50()
resnet = train_two_phase_model(
    resnet,
    [resnet.classifier],
    model_name="ResNet50",
    checkpoint_name="resnet_best.pt",
    batch_size=64,
    phase1_lr=1e-4,
    phase2_lr=1e-4,
)


In [ ]:
densenet = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
densenet.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(1024, len(CLASS_NAMES)),
)
densenet = train_two_phase_model(
    densenet,
    [densenet.classifier],
    model_name="DenseNet121",
    checkpoint_name="densenet_best.pt",
    batch_size=256,
    phase1_lr=5e-4,
    phase2_lr=5e-5,
)


In [ ]:
efficientnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
in_features = efficientnet.classifier[1].in_features
efficientnet.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_features, len(CLASS_NAMES)),
)
efficientnet = train_two_phase_model(
    efficientnet,
    [efficientnet.classifier],
    model_name="EfficientNet-B0",
    checkpoint_name="efficientnet_best.pt",
    batch_size=256,
    phase1_lr=1e-3,
    phase2_lr=5e-5,
)


In [ ]:
vit = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)
vit.heads.head = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(vit.heads.head.in_features, len(CLASS_NAMES)),
)
vit = train_two_phase_model(
    vit,
    [vit.heads.head],
    model_name="ViT-B/16",
    checkpoint_name="vit_best.pt",
    batch_size=64,
    phase1_lr=1e-4,
    phase2_lr=1e-5,
)


In [ ]:
mobilenet = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)
in_features = mobilenet.classifier[3].in_features
mobilenet.classifier[3] = nn.Linear(in_features, len(CLASS_NAMES))
mobilenet = train_two_phase_model(
    mobilenet,
    [mobilenet.classifier],
    model_name="MobileNetV3-Large",
    checkpoint_name="mobilenet_best.pt",
    batch_size=256,
    phase1_lr=1e-3,
    phase2_lr=1e-4,
)


The checkpoint files generated by this notebook are saved in `model_checkpoints/` and are loaded by `Project_Report.ipynb`.
